# NBA 3.1 Feature Engineering: Competitor Drug NBRx Share (Notebook 31)

**Purpose:** Compute HCP-level Kerendia NBRx market share vs competitors across SGLT2i, MRA, and HF drug classes, using V6 LAAD LRx table with TREATMENT_CATEGORY from the final product mapping.

**Key changes from NBA 3.0:**
- Add Inpefa to SGLT2i competitor set (5 competitors: Farxiga, Invokana, Jardiance, Steglatro, Inpefa)
- Add MRA competitor share (Kerendia / (Kerendia + Spironolactone + Eplerenone))
- Add HF drug share (Kerendia / (Kerendia + Entresto + Verquvo + Inpefa + Farxiga + Jardiance))
- Switch from NPI_NUMBER to PROVIDER_ID (aligns with notebooks 26/27/28)
- Add 1, 3, 6-month windows (currently only 12-month)
- Use step2_fill_missing and step3_lag from 00_step23_function (same as NBA 3.0)

**Data source:** V6 LAAD LRx table (same as NBA 3.0)
**Output:** `kerendia_nba_3_1_volume/notebook_output/hcp_features/competitor_share_features/`

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_config"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_step23_function"

In [0]:
# Cell 1: Configuration
import gc
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, expr, concat, substring, trunc

V6_TABLE = "KERENDIA_LAAD_BASED_LRX_V6"
VOLUME_BASE = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume"
HCP_UNIVERSE_PATH = f"{VOLUME_BASE}/HCP_Universe_without_call_hcps"
OUTPUT_PATH = f"{VOLUME_BASE}/notebook_output/hcp_features/competitor_share_features"

HCP_KEY = "PROVIDER_ID"  # Switched from NPI_NUMBER (decision #11)
LAG_WINDOWS = [1, 3, 6, 12]

# SGLT2i competitors: 5 (added Inpefa)
SGLT2I_GROUPS = ["FARXIGA", "INVOKANA", "JARDIANCE", "STEGALTRO", "INPEFA"]
# MRA competitors: Kerendia vs Spironolactone/Eplerenone
MRA_GROUPS = ["KERENDIA", "MINERALOCORTICOID RECEPTOR ANTAGONISTS"]
# HF drug competitors: Kerendia, Entresto, Verquvo, Inpefa, Farxiga, Jardiance
HF_DRUG_GROUPS = ["KERENDIA", "ENTRESTO", "VERQUVO", "INPEFA", "FARXIGA", "JARDIANCE"]

all_competitor_groups = list(set(SGLT2I_GROUPS + MRA_GROUPS + HF_DRUG_GROUPS))
print(f"Competitor groups: {len(all_competitor_groups)}")
print(f"SGLT2i: {len(SGLT2I_GROUPS)}, MRA: {len(MRA_GROUPS)}, HF drug: {len(HF_DRUG_GROUPS)}")
from pyspark.storagelevel import StorageLevel

dbutils.widgets.text("test_mode", "false")
TEST_MODE = dbutils.widgets.get("test_mode") == "true"
TEST_HCP_COUNT = 1000
print(f"Key: {HCP_KEY}, Windows: {LAG_WINDOWS}")
print(f"Output: {OUTPUT_PATH}")
print(f"Test mode: {TEST_MODE}, Test HCP count: {TEST_HCP_COUNT}")

In [0]:
# Cell 2: Read V6, filter to HCP universe, date processing
# OPTIMIZED: date filter in SQL, broadcast join, cache, minimal count() calls
hcp_universe = spark.read.parquet(HCP_UNIVERSE_PATH).select("PROVIDER_ID")
if TEST_MODE:
    hcp_universe = hcp_universe.limit(TEST_HCP_COUNT)
    print(f"TEST MODE: limiting to {TEST_HCP_COUNT} HCPs")

query = f"""
    SELECT DISTINCT WEEK_ENDING_DATE, PROVIDER_ID, PERIOD_ID, PRODUCT_GROUP, NBRX, NPI, PRI_SPCL_DESC
    from `ph-com-ai-us-prod-catalog-2553213839599380`.`kerendia_laad_based_lrx`.{V6_TABLE}
"""
kerendia_claims = spark.sql(query)
kerendia_claims = F.broadcast(hcp_universe).join(kerendia_claims, on="PROVIDER_ID", how="inner")

kerendia_claims = kerendia_claims.withColumn("SVC_DT_M", F.trunc(col("PERIOD_ID"), 'month')).drop('WEEK_ENDING_DATE', 'NPI', 'PRI_SPCL_DESC')
kerendia_claims.persist(StorageLevel.MEMORY_AND_DISK)
materialized_count = kerendia_claims.count()
print(f"Cached V6 (filtered): {materialized_count:,} rows")
index_cols = ["PROVIDER_ID", "SVC_DT_M"]

In [0]:
# Cell 3: Pivot by PRODUCT_GROUP, consolidate FARXIGA, select competitors, compute totals
from functools import reduce
# Consolidate FARXIGA (brand + generic -> FARXIGA), same as NBA 3.0
df_category_claims = kerendia_claims.withColumn("PRODUCT_GROUP", when(col("PRODUCT_GROUP").like("%FARXIGA%"), "FARXIGA").otherwise(col("PRODUCT_GROUP")))
df_category_claims = df_category_claims.withColumn("PRODUCT_GROUP", when((col("PRODUCT_GROUP").isNull()) | (col("PRODUCT_GROUP") == 'NULL'), 'UNKNOWN').otherwise(col("PRODUCT_GROUP")))

# Pivot by PRODUCT_GROUP, sum NBRX
df_category = df_category_claims.groupBy(index_cols).pivot("PRODUCT_GROUP").agg(F.sum("NBRX").alias("SUM_NBRX"))

# Clean column names (replace spaces, special chars for Delta compatibility)
def clean_col_name(c):
    return c.replace(", ", " ").replace("+", "and").replace("/", " or ").replace("-", "_").replace(" ", "_").replace("__", "_").replace("(", "").replace(")", "").replace("&", "").upper()

df_category = df_category.select([F.col(c).alias(clean_col_name(c)) if c not in index_cols else col(c) for c in df_category.columns])

# Select only competitor columns (SGLT2i + MRA + HF drug)
# Use cleaned group names for matching with cleaned column names
cleaned_competitor_groups = [clean_col_name(c) for c in all_competitor_groups]
select_expr = index_cols + [c for c in df_category.columns if any(cg in c for cg in cleaned_competitor_groups)]
df_category_new = df_category.select(*select_expr)

# Compute SGLT2i total
cleaned_sglt2i = [clean_col_name(sg) for sg in SGLT2I_GROUPS if sg != "FARXIGA"]
glt2i_cols = [c for c in df_category_new.columns if any(sg in c for sg in cleaned_sglt2i) or "FARXIGA" in c]
if glt2i_cols:
    df_category_new = df_category_new.withColumn('TOTAL_NBRX_CLAIMS_SGLT2I', reduce(lambda a, b: a + b, [F.coalesce(col(c), F.lit(0)) for c in glt2i_cols]))

# Compute MRA total
cleaned_mra = [clean_col_name(mg) for mg in MRA_GROUPS]
mra_cols = [c for c in df_category_new.columns if any(mg in c for mg in cleaned_mra)]
if mra_cols:
    df_category_new = df_category_new.withColumn('TOTAL_NBRX_CLAIMS_MRA', reduce(lambda a, b: a + b, [F.coalesce(col(c), F.lit(0)) for c in mra_cols]))

# Compute HF drug total
cleaned_hf = [clean_col_name(hg) for hg in HF_DRUG_GROUPS]
hf_cols = [c for c in df_category_new.columns if any(hg in c for hg in cleaned_hf)]
if hf_cols:
    df_category_new = df_category_new.withColumn('TOTAL_NBRX_CLAIMS_HF_DRUG', reduce(lambda a, b: a + b, [F.coalesce(col(c), F.lit(0)) for c in hf_cols]))

print(f"Competitor data: {df_category_new.count():,} rows, {len(df_category_new.columns)} cols")

In [0]:
# Cell 4: step2_fill_missing and step3_lag with [1, 3, 6, 12]
# OPTIMIZED: removed count() call
df_nbrx_shares = step2_fill_missing(df_category_new, key_col="PROVIDER_ID", date_col="SVC_DT_M")
keep_cols = ["PROVIDER_ID", "SVC_DT_M"]
cols_to_drop = [c for c in df_nbrx_shares.columns if c not in keep_cols]
df_nbrx_feats = step3_lag(df_nbrx_shares, key_col="PROVIDER_ID", date_col="SVC_DT_M", last_i_month_list=LAG_WINDOWS)
del df_nbrx_shares; gc.collect()
df_nbrx_feats = df_nbrx_feats.drop(*cols_to_drop).filter(df_nbrx_feats.PROVIDER_ID.isNotNull()).fillna(0)
print(f"After step3_lag: {len(df_nbrx_feats.columns)} cols")

In [0]:
# Cell 5: Compute share features for all 3 groups x 4 windows
# SGLT2i shares: PERC_{DRUG}_NBRX_SHARE_{WINDOW}M = drug / total_SGLT2i
# MRA share: PERC_KERENDIA_MRA_SHARE_{WINDOW}M = Kerendia / total_MRA
# HF drug share: PERC_KERENDIA_HF_DRUG_SHARE_{WINDOW}M = Kerendia / total_HF_drug

for lag_m in LAG_WINDOWS:
    # SGLT2i shares (per competitor)
    total_col = f"TOTAL_NBRX_CLAIMS_SGLT2I_IN_LAST_{lag_m}_MONTH"
    if total_col in df_nbrx_feats.columns:
        for drug in SGLT2I_GROUPS:
            drug_col = None
            for c in df_nbrx_feats.columns:
                if drug in c and f"IN_LAST_{lag_m}_MONTH" in c and "TOTAL_NBRX_CLAIMS" in c and "SGLT2I" not in c:
                    drug_col = c
                    break
            if drug_col:
                share_col = f"PERC_{drug}_NBRX_SHARE_{lag_m}M"
                df_nbrx_feats = df_nbrx_feats.withColumn(share_col, when(col(total_col) != 0, col(drug_col) / col(total_col)).otherwise(0))
    
    # MRA share (Kerendia's share of MRA market)
    mra_total_col = f"TOTAL_NBRX_CLAIMS_MRA_IN_LAST_{lag_m}_MONTH"
    kerendia_col = None
    for c in df_nbrx_feats.columns:
        if "KERENDIA" in c and f"IN_LAST_{lag_m}_MONTH" in c and "MRA" not in c and "HF_DRUG" not in c:
            kerendia_col = c
            break
    if mra_total_col in df_nbrx_feats.columns and kerendia_col:
        df_nbrx_feats = df_nbrx_feats.withColumn(f"PERC_KERENDIA_MRA_SHARE_{lag_m}M", when(col(mra_total_col) != 0, col(kerendia_col) / col(mra_total_col)).otherwise(0))
    
    # HF drug share (Kerendia's share of HF drug market)
    hf_total_col = f"TOTAL_NBRX_CLAIMS_HF_DRUG_IN_LAST_{lag_m}_MONTH"
    if hf_total_col in df_nbrx_feats.columns and kerendia_col:
        df_nbrx_feats = df_nbrx_feats.withColumn(f"PERC_KERENDIA_HF_DRUG_SHARE_{lag_m}M", when(col(hf_total_col) != 0, col(kerendia_col) / col(hf_total_col)).otherwise(0))

print(f"Share features computed. Total columns: {len(df_nbrx_feats.columns)}")

In [0]:
# Cell 6: Filter and save
# OPTIMIZED: removed count() call
df_nbrx_feats = df_nbrx_feats.filter(F.to_date('SVC_DT_M') >= '2022-11-01')
curr_date = str(dt.today().strftime('%Y_%m_%d'))
df_nbrx_feats = df_nbrx_feats.withColumn("PULL_DT", F.lit(curr_date))
print(f"Saving to: {OUTPUT_PATH}")
df_nbrx_feats.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(OUTPUT_PATH)
print(f"Notebook 31 (Competitor Share) complete. Saved to NBA 3.1 volume.")